# LeetCode 555 — Split Concatenated Strings bằng Python

Link đề: https://leetcode.com/problems/split-concatenated-strings/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, String

Skeleton LeetCode:

```python
class Solution:
    def splitLoopedString(self, strs: List[str]) -> str:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí để xem đề gốc). Đề trong notebook được viết lại theo hiểu biết về bài và theo ví dụ đã biết; **nên đối chiếu lại đề gốc nếu bạn có tài khoản Premium**, đặc biệt phần ví dụ và ràng buộc.

Ý chính: ta có một dãy chuỗi, được phép **đảo ngược** mỗi chuỗi (hoặc không), nối chúng theo đúng thứ tự thành một **vòng tròn**, rồi **cắt vòng ở một điểm** để được chuỗi thường. Cần chuỗi **lớn nhất theo thứ tự từ điển**. Notebook đi từ cách vét cạn `2^n` kiểu đảo, qua lời giải tham lam (greedy) chuẩn, rồi hai cách tối ưu hóa/diễn đạt khác (cắt tỉa theo ký tự lớn nhất, nhân đôi vòng tròn để cắt cửa sổ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi chuỗi trong `strs` được chọn **giữ nguyên hoặc đảo ngược**; nối theo thứ tự thành **vòng tròn**; cắt vòng ở một chỗ bất kỳ; trả về chuỗi **lớn nhất theo thứ tự từ điển** trong tất cả các khả năng.
2. Nắm các công cụ Python/chuỗi: đảo chuỗi `s[::-1]`, so sánh chuỗi theo thứ tự từ điển (`<`, `max`), cắt lát `s[j:]` và `s[:j]`, `''.join`, xoay vòng một chuỗi bằng `s[c:] + s[:c]`, và `itertools.product` để liệt kê các tổ hợp đảo/không đảo.
3. Đi từ **vét cạn `2^n` kiểu đảo × mọi điểm cắt** → **chuẩn hóa (mỗi chuỗi lấy bản lớn hơn) + thử mọi điểm cắt trong từng chuỗi ở cả hai chiều** → **cắt tỉa: chỉ cắt tại ký tự lớn nhất** → **nhân đôi vòng tròn và cắt cửa sổ**.
4. Hiểu mấu chốt tham lam: khi đã biết **chuỗi nào chứa điểm cắt và chiều của nó**, độ dài kết quả cố định và mỗi chuỗi còn lại chiếm **một đoạn vị trí cố định**, nên mỗi chuỗi còn lại độc lập chọn chiều **lớn hơn** (`max(s, s[::-1])`).
5. Tránh các lỗi kinh điển: quên chuẩn hóa các chuỗi còn lại, quên thử chiều còn lại của chuỗi chứa điểm cắt, quên đoạn "vòng lại" `strs[:i]`, chỉ cắt ở ranh giới giữa các chuỗi, và dùng `reversed(s)` (trả về iterator, không phải chuỗi) để so sánh.

## 2. Đề bài & ví dụ

Cho một mảng chuỗi `strs`. Bạn có thể nối các chuỗi lại thành một **vòng tròn** (loop) theo **đúng thứ tự cho trước**; với mỗi chuỗi bạn được **chọn đảo ngược hoặc không**. Trong tất cả các vòng có thể tạo ra, hãy **cắt vòng tại một điểm bất kỳ** để được một chuỗi thường (bắt đầu từ ký tự ngay sau điểm cắt, đi hết một vòng).

Trả về **chuỗi lớn nhất theo thứ tự từ điển** có thể thu được.

Cụ thể hai giai đoạn:

1. Nối mọi chuỗi thành vòng, mỗi chuỗi giữ nguyên hoặc đảo, thứ tự nối giữ nguyên.
2. Cắt vòng tại một điểm, chuỗi thu được bắt đầu từ ký tự ngay sau điểm cắt. Lấy chuỗi lớn nhất trong mọi khả năng.

**Ví dụ 1**

- Input: `strs = ["abc","xyz"]`
- Output: `"zyxcba"`
- Giải thích: các vòng có thể: `-abcxyz-`, `-abczyx-`, `-cbaxyz-`, `-cbazyx-` (dấu `-` đánh dấu chỗ nối đầu cuối của vòng). Chuỗi lớn nhất đến từ vòng thứ tư `cbazyx`: cắt ngay trước chữ `z` (đầu đoạn `zyx`) thì đọc một vòng được `zyxcba`.

**Ví dụ 2**

- Input: `strs = ["abc"]`
- Output: `"cba"`

**Ràng buộc:**

- `1 <= strs.length <= 1000`
- `1 <= strs[i].length <= 1000`
- `1 <= sum(strs[i].length) <= 1000`
- `strs[i]` chỉ gồm chữ cái tiếng Anh thường (`a`..`z`)

Chú ý: tổng độ dài chỉ tối đa `1000`; ký hiệu `L` sẽ dùng cho tổng độ dài, `n` là số chuỗi.

In [1]:
from typing import List
from itertools import product
import random
import time

examples = [
    (["abc", "xyz"], "zyxcba"),
    (["abc"], "cba"),
]
for strs, expected in examples:
    print("n =", len(strs), "| tổng độ dài L =", sum(map(len, strs)), "| kỳ vọng:", repr(expected))

n = 2 | tổng độ dài L = 6 | kỳ vọng: 'zyxcba'
n = 1 | tổng độ dài L = 3 | kỳ vọng: 'cba'


## 3. Kiến thức nền

### 3.1. Đảo chuỗi bằng `s[::-1]`

Cú pháp cắt lát `s[start:stop:step]` với `step = -1` đi **ngược** từ cuối chuỗi về đầu, nên `s[::-1]` là bản đảo của `s`. Chuỗi trong Python là bất biến (immutable): `s[::-1]` tạo **chuỗi mới**, không đổi `s`.

In [2]:
s = "abc"
print(s[::-1], "| s vẫn là", s)
print("ab"[::-1], "a"[::-1], ""[::-1] == "")
palin = "aba"
print(palin[::-1] == palin)          # chuỗi đối xứng: đảo xong không đổi

cba | s vẫn là abc
ba a True
True


### 3.2. So sánh chuỗi theo thứ tự từ điển

Python so sánh hai chuỗi bằng cách so **từng ký tự từ trái sang phải** theo mã ký tự (với chữ thường `a < b < ... < z`). Gặp vị trí đầu tiên khác nhau thì ký tự lớn hơn quyết định chuỗi nào lớn hơn; nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi dài hơn lớn hơn. `max(x, y)` trên chuỗi dùng cùng quy tắc, nên `max(s, s[::-1])` cho **bản lớn hơn** của `s` và bản đảo.

In [3]:
print("abd" > "abc", "b" > "abc", "ab" < "abc", "abc" < "abd")
print(max("abc", "cba"))            # cba
print(max("acb", "bca"))            # bca
print(max("aba", "aba"))            # bằng nhau
print(sorted(["b", "ab", "a", "ba"]))

True True True True
cba
bca
aba
['a', 'ab', 'b', 'ba']


### 3.3. Cắt lát `s[j:]` và `s[:j]`

`s[:j]` là **j ký tự đầu** (tiền tố), `s[j:]` là phần **còn lại từ vị trí j** (hậu tố). Ghép lại `s[:j] + s[j:] == s` với mọi `j`. Đây là hai mảnh khi ta "cắt" chuỗi `s` tại điểm `j`.

In [4]:
s = "abcde"
for j in range(len(s)):
    print(j, repr(s[:j]), repr(s[j:]), s[:j] + s[j:] == s)

0 '' 'abcde' True
1 'a' 'bcde' True
2 'ab' 'cde' True
3 'abc' 'de' True
4 'abcd' 'e' True


### 3.4. Vòng tròn và phép xoay `s[c:] + s[:c]`

Cắt một **vòng tròn** các ký tự tại điểm `c` rồi đọc hết một vòng chính là `loop[c:] + loop[:c]` (phép xoay vòng, *rotation*). Một chuỗi dài `L` có đúng `L` cách cắt (`c = 0..L-1`).

In [5]:
loop = "cbazyx"
rotations = [loop[c:] + loop[:c] for c in range(len(loop))]
print(rotations)
print("lớn nhất:", max(rotations))

['cbazyx', 'bazyxc', 'azyxcb', 'zyxcba', 'yxcbaz', 'xcbazy']
lớn nhất: zyxcba


### 3.5. `''.join` — nối nhiều chuỗi

`''.join(list_of_strings)` nối các chuỗi trong danh sách thành **một** chuỗi (dấu `''` là chuỗi phân cách rỗng). Nối bằng `join` hiệu quả hơn cộng `+` lặp lại trong vòng lặp. Cắt lát danh sách `strs[i+1:]`, `strs[:i]` lấy "phần sau" và "phần trước" `strs[i]`.

In [6]:
strs = ["ab", "cd", "ef", "gh"]
i = 1
print(strs[i + 1:], strs[:i])
print("".join(strs[i + 1:]) + "".join(strs[:i]))   # các chuỗi còn lại, theo thứ tự vòng, bắt đầu ngay sau strs[i]

['ef', 'gh'] ['ab']
efghab


### 3.6. `itertools.product` — liệt kê mọi tổ hợp đảo/không đảo

`product((False, True), repeat=n)` sinh mọi bộ `n` giá trị đúng/sai, tức `2^n` tổ hợp, mỗi bộ mô tả "chuỗi nào bị đảo". Ta dùng nó cho cách vét cạn (cũng là **oracle** để kiểm chứng ở mục 12).

In [7]:
flips_list = list(product((False, True), repeat=3))
print(len(flips_list), "tổ hợp:")
for flips in flips_list:
    print(flips)

8 tổ hợp:
(False, False, False)
(False, False, True)
(False, True, False)
(False, True, True)
(True, False, False)
(True, False, True)
(True, True, False)
(True, True, True)


## 4. Ý tưởng nền tảng / trực giác

Hãy tự làm tay ví dụ 1 `["abc", "xyz"]`. Có `2^2 = 4` cách chọn chiều và với mỗi vòng có `L = 6` điểm cắt, tổng cộng `4 × 6 = 24` chuỗi ứng viên. Ta liệt kê hết và lấy lớn nhất:

In [8]:
strs = ["abc", "xyz"]
all_candidates = []
for flips in product((False, True), repeat=len(strs)):
    parts = [s[::-1] if f else s for s, f in zip(strs, flips)]
    loop = "".join(parts)
    cuts = [loop[c:] + loop[:c] for c in range(len(loop))]
    print(flips, "vòng", loop, "-> lớn nhất:", max(cuts))
    all_candidates.extend(cuts)
print("tổng số ứng viên:", len(all_candidates), "| lớn nhất:", max(all_candidates))

(False, False) vòng abcxyz -> lớn nhất: zabcxy
(False, True) vòng abczyx -> lớn nhất: zyxabc
(True, False) vòng cbaxyz -> lớn nhất: zcbaxy
(True, True) vòng cbazyx -> lớn nhất: zyxcba
tổng số ứng viên: 24 | lớn nhất: zyxcba


Quan sát:

- **Thứ tự các chuỗi không đổi.** Chỉ có chiều từng chuỗi và điểm cắt thay đổi.
- Chuỗi kết quả luôn dài đúng `L` (tổng độ dài), nên **các ứng viên có cùng độ dài**: so sánh từ điển giữa chúng đơn giản là so từng ký tự từ trái sang.
- Kết quả tốt nhất `zyxcba` có `zyx` (chuỗi `xyz` đảo) và `cba` (chuỗi `abc` đảo): **cả hai chuỗi đều được đảo về bản lớn hơn**, và điểm cắt nằm ngay **đầu** chuỗi `zyx`.

Câu hỏi dẫn đường: số ứng viên `2^n · L` bùng nổ theo `n` (với `n = 1000` là không thể). Có cần thử hết `2^n` kiểu đảo? Gợi ý: hầu hết các chuỗi *không chứa điểm cắt* nên chúng chỉ là "khối" nằm cố định trong kết quả. Nên đảo hay không có thể quyết định **riêng từng khối**. Mục 6 làm rõ điều này.

## 5. Cách 1 — Brute Force: vét cạn mọi kiểu đảo và mọi điểm cắt

**Ý tưởng.** Làm đúng như đề: duyệt `2^n` tổ hợp đảo; với mỗi tổ hợp, nối thành vòng, thử mọi điểm cắt `c = 0..L-1`, lấy chuỗi lớn nhất. Cách này **đúng hiển nhiên** nên cũng dùng làm *oracle* để kiểm tra các cách còn lại (chỉ với `n` nhỏ).

In [9]:
def split_brute(strs):
    best = ""
    for flips in product((False, True), repeat=len(strs)):
        loop = "".join(s[::-1] if f else s for s, f in zip(strs, flips))
        for c in range(len(loop)):
            best = max(best, loop[c:] + loop[:c])
    return best


for strs, expected in examples:
    got = split_brute(strs)
    print(repr(got), "(kỳ vọng", repr(expected), ")")
    assert got == expected

'zyxcba' (kỳ vọng 'zyxcba' )
'cba' (kỳ vọng 'cba' )


**Độ phức tạp:** Time `O(2^n · L^2)` (mỗi vòng có `L` điểm cắt, mỗi lần dựng chuỗi tốn `O(L)`). Space `O(L)`.

**Nút thắt:** `n` tới `1000` thì `2^n` là con số khổng lồ. Cần chứng minh được rằng **không cần thử từng tổ hợp đảo**, mà mỗi chuỗi có thể chọn chiều theo một quy tắc đơn giản.

## 6. Ý tưởng thuật toán / bước đột phá

Giả sử ta **đã chọn** điểm cắt nằm trong chuỗi `strs[i]` (chiều `o` đã chọn: `o = strs[i]` hoặc `o = strs[i][::-1]`) tại vị trí `j` trong `o`. Khi đó chuỗi kết quả luôn có dạng:

```
kết quả = o[j:]  +  (strs[i+1] ... strs[n-1] strs[0] ... strs[i-1])  +  o[:j]
          ^ đoạn 1       ^ các chuỗi còn lại theo thứ tự vòng        ^ đoạn 3
```

**Quan sát then chốt.** Mỗi chuỗi còn lại (`strs[k]`, `k ≠ i`) chiếm **một đoạn vị trí liên tiếp cố định** trong kết quả, độ dài đúng bằng `len(strs[k])`, bất kể ta chọn chiều nào cho nó. Vì các đoạn không chồng lên nhau và có vị trí cố định, nên để kết quả **lớn nhất**, ta có thể cho **từng đoạn lớn nhất độc lập**: mỗi chuỗi còn lại chọn `max(s, s[::-1])`.

**Hệ quả:**

1. **Chuẩn hóa** mọi chuỗi: `s = max(s, s[::-1])` cho các chuỗi **không chứa điểm cắt**.
2. Với **chuỗi chứa điểm cắt**, chiều của nó và vị trí cắt `j` ảnh hưởng lẫn nhau (cùng quyết định ký tự đầu), nên ta **thử cả hai chiều** và **mọi** `j`.
3. Duyệt `i = 0..n-1` (chuỗi chứa điểm cắt) và `j` trong chuỗi đó; tổng số ứng viên là `2 · L` (hai chiều × `L` điểm cắt). Mỗi ứng viên dựng trong `O(L)` nên tổng `O(L^2)`.

Công thức một ứng viên với chuỗi `s = strs[i]` (đã chuẩn hóa hay chưa đều thử cả hai chiều) và `t` là phần ghép các chuỗi còn lại đã chuẩn hóa:

```
chiều giữ nguyên : s[j:] + t + s[:j]
chiều đảo        : s_dao[j:] + t + s_dao[:j]    (với s_dao = s[::-1])
```

## 7. Các cách tiếp theo

Phần này có thêm 3 cách nữa (tổng 4 cách): tham lam chuẩn (chuẩn hóa + thử điểm cắt), cắt tỉa theo ký tự lớn nhất, và nhân đôi vòng tròn rồi cắt cửa sổ.

### 7.1. Cách 2 — Tham lam chuẩn: chuẩn hóa + thử mọi điểm cắt trong từng chuỗi

**Giải thích từng đoạn.**

1. `norm = [max(s, s[::-1]) for s in strs]`: mỗi chuỗi lấy bản lớn hơn. Đây là giá trị mà một chuỗi "đứng yên trong khối" nên có.
2. Với mỗi chuỗi `i`, tạo `t` là phần các chuỗi **còn lại** (đã chuẩn hóa) theo thứ tự vòng: `norm[i+1:] + norm[:i]`.
3. Với chuỗi chứa điểm cắt `s = norm[i]`, thử **cả hai chiều** của nó: chiều giữ nguyên (`s[j:] + t + s[:j]`) và chiều đảo; vì `norm[i]` chỉ là một trong hai chiều của `strs[i]`, hai chiều này phủ đủ mọi khả năng.
4. Giữ chuỗi lớn nhất.

*(Cách viết gọn: với `a = s[j:]`, `b = s[:j]` thì hai ứng viên là `a + t + b` và `b[::-1] + t + a[::-1]`. Ứng viên thứ hai chính là cắt chuỗi đảo `s[::-1]` tại điểm tương ứng: `s[::-1] = a[::-1] + b[::-1]`, cắt giữa hai nửa cho `b[::-1] + t + a[::-1]`.)*

In [10]:
def split_greedy(strs):
    norm = [max(s, s[::-1]) for s in strs]
    best = "".join(norm)                          # ứng viên nền: cắt tại đầu chuỗi 0, mọi chuỗi chuẩn hóa
    for i, s in enumerate(norm):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            best = max(best, a + t + b)           # giữ chiều của s (đã chuẩn hóa)
            best = max(best, b[::-1] + t + a[::-1])   # chiều đảo của s
    return best


for strs, expected in examples:
    got = split_greedy(strs)
    print(repr(got), "(kỳ vọng", repr(expected), ")")
    assert got == expected

'zyxcba' (kỳ vọng 'zyxcba' )
'cba' (kỳ vọng 'cba' )


In [11]:
class Solution:
    def splitLoopedString(self, strs: List[str]) -> str:
        strs = [max(s, s[::-1]) for s in strs]
        ans = "".join(strs)
        for i, s in enumerate(strs):
            t = "".join(strs[i + 1:]) + "".join(strs[:i])
            for j in range(len(s)):
                a, b = s[j:], s[:j]
                ans = max(ans, a + t + b)
                ans = max(ans, b[::-1] + t + a[::-1])
        return ans

print(Solution().splitLoopedString(["abc", "xyz"]), Solution().splitLoopedString(["abc"]))

zyxcba cba


**Độ phức tạp:** Time `O(L^2)` (có `2L` ứng viên, dựng mỗi ứng viên `O(L)`; thêm `n` lần dựng `t` tốn `O(n · L)` nhưng `n ≤ L`). Với `L ≤ 1000` chỉ khoảng vài triệu thao tác ký tự. Space `O(L)`.

### 7.2. Cách 3 — Cắt tỉa: chỉ cắt tại ký tự lớn nhất toàn cục

**Giải thích từng đoạn.** Chuỗi kết quả tốt nhất **phải bắt đầu bằng ký tự lớn nhất** `top` có trong toàn bộ `strs` (ta luôn có thể chọn chiều và điểm cắt để đưa một ký tự `top` lên đầu; ví dụ cắt ngay tại nó, chiều tùy ý). Do đó ứng viên nào bắt đầu bằng ký tự khác `top` đều **chắc chắn thua**, ta bỏ qua mà không cần dựng chuỗi. Chỉ dựng ứng viên khi `o[j] == top`. Nhiều trường hợp (chuỗi ngẫu nhiên, bảng chữ cái lớn) loại được gần hết ứng viên.

In [12]:
def split_pruned(strs):
    norm = [max(s, s[::-1]) for s in strs]
    top = max(max(s) for s in strs)               # ký tự lớn nhất toàn bộ (đảo chuỗi không đổi tập ký tự)
    best = ""
    built = 0
    for i, s in enumerate(strs):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        for o in (s, s[::-1]):                    # cả hai chiều của chuỗi chứa điểm cắt
            for j, ch in enumerate(o):
                if ch != top:
                    continue                      # ứng viên không bắt đầu bằng top: bỏ
                built += 1
                best = max(best, o[j:] + t + o[:j])
    split_pruned.built = built
    return best


for strs, expected in examples:
    got = split_pruned(strs)
    print(repr(got), "| số ứng viên phải dựng:", split_pruned.built, "(kỳ vọng", repr(expected), ")")
    assert got == expected

'zyxcba' | số ứng viên phải dựng: 2 (kỳ vọng 'zyxcba' )
'cba' | số ứng viên phải dựng: 2 (kỳ vọng 'cba' )


In [13]:
class Solution:
    def splitLoopedString(self, strs: List[str]) -> str:
        norm = [max(s, s[::-1]) for s in strs]
        top = max(max(s) for s in strs)
        ans = ""
        for i, s in enumerate(strs):
            t = "".join(norm[i + 1:]) + "".join(norm[:i])
            for o in (s, s[::-1]):
                for j, ch in enumerate(o):
                    if ch == top:
                        ans = max(ans, o[j:] + t + o[:j])
        return ans

print(Solution().splitLoopedString(["abc", "xyz"]), Solution().splitLoopedString(["abc"]))

zyxcba cba


**Độ phức tạp:** Time trường hợp xấu vẫn `O(L^2)` (khi mọi ký tự đều bằng `top`, ví dụ toàn `'z'`), nhưng trung bình ít hơn nhiều vì chỉ dựng các ứng viên bắt đầu bằng `top`. Space `O(L)`.

Cắt tỉa này là một **tối ưu hằng số**, không đổi bậc độ phức tạp. Bài này không cần nó để qua giới hạn `L ≤ 1000`, nhưng ý nghĩ "kết quả tốt nhất phải bắt đầu bằng ký tự lớn nhất" rất đáng nhớ.

### 7.3. Cách 4 — Nhân đôi vòng tròn rồi cắt cửa sổ

**Giải thích từng đoạn.** Với một vòng tròn `loop` dài `L`, mọi chuỗi cắt từ điểm `st` là cửa sổ `(loop + loop)[st : st + L]`. Vậy thay vì dựng `o[j:] + t + o[:j]`, ta:

1. với mỗi chuỗi `i` và chiều `o` của nó, dựng một lần **vòng** `loop = norm[:i] + o + norm[i+1:]` (các chuỗi khác đã chuẩn hóa), rồi `doubled = loop + loop`;
2. điểm cắt nằm trong chuỗi `i` ứng với `st = len(tiền tố) + j` với `j = 0..len(o)-1`; cửa sổ `doubled[st : st + L]` là ứng viên.

Cách nhìn này gọn: "xoay vòng = cắt cửa sổ trên chuỗi nhân đôi", và mỗi (chuỗi, chiều) chỉ dựng chuỗi lớn một lần.

In [14]:
def split_doubled(strs):
    norm = [max(s, s[::-1]) for s in strs]
    L = sum(map(len, strs))
    best = ""
    prefix_len = 0                                  # tổng độ dài các chuỗi trước i
    for i, s in enumerate(strs):
        before = "".join(norm[:i])
        after = "".join(norm[i + 1:])
        for o in (s, s[::-1]):
            loop = before + o + after
            doubled = loop + loop
            for j in range(len(o)):
                st = prefix_len + j
                best = max(best, doubled[st:st + L])
        prefix_len += len(s)
    return best


for strs, expected in examples:
    got = split_doubled(strs)
    print(repr(got), "(kỳ vọng", repr(expected), ")")
    assert got == expected

'zyxcba' (kỳ vọng 'zyxcba' )
'cba' (kỳ vọng 'cba' )


In [15]:
class Solution:
    def splitLoopedString(self, strs: List[str]) -> str:
        norm = [max(s, s[::-1]) for s in strs]
        total = sum(map(len, strs))
        ans = ""
        offset = 0
        for i, s in enumerate(strs):
            before = "".join(norm[:i])
            after = "".join(norm[i + 1:])
            for o in (s, s[::-1]):
                loop = before + o + after
                doubled = loop + loop
                for j in range(len(o)):
                    ans = max(ans, doubled[offset + j: offset + j + total])
            offset += len(s)
        return ans

print(Solution().splitLoopedString(["abc", "xyz"]), Solution().splitLoopedString(["abc"]))

zyxcba cba


**Độ phức tạp:** Time `O(L^2)` (mỗi cửa sổ cắt lát tốn `O(L)`, có `2L` cửa sổ; thêm `O(n · L)` dựng vòng). Space `O(L)`.

*Lưu ý nhỏ:* cách này cùng bậc với cách 2, nhưng thể hiện rõ "vòng tròn". Cả ba cách 2, 3, 4 đều dựa trên cùng **lập luận tham lam** (mục 6); cách 1 là vét cạn độc lập để kiểm chứng.

## 8. TRACE từng bước

Trace cách 2 (tham lam chuẩn) với `strs = ["abc", "xyz"]`, kỳ vọng `"zyxcba"`.

**Bước chuẩn hóa:** `"abc"` đảo là `"cba"` (lớn hơn) → `"cba"`; `"xyz"` đảo là `"zyx"` (lớn hơn) → `"zyx"`. `norm = ["cba", "zyx"]`. Ứng viên nền `best = "cbazyx"`.

| `i` | `s` | `t` | `j` | `a = s[j:]` | `b = s[:j]` | giữ chiều `a+t+b` | đảo chiều `b[::-1]+t+a[::-1]` | `best` sau bước |
|---|---|---|---|---|---|---|---|---|
| 0 | cba | zyx | 0 | cba | (rỗng) | cbazyx | zyxabc | zyxabc |
| 0 | cba | zyx | 1 | ba | c | bazyxc | czyxab | zyxabc |
| 0 | cba | zyx | 2 | a | cb | azyxcb | bczyxa | zyxabc |
| 1 | zyx | cba | 0 | zyx | (rỗng) | zyxcba | cbaxyz | zyxcba |
| 1 | zyx | cba | 1 | yx | z | yxcbaz | zcbaxy | zyxcba |
| 1 | zyx | cba | 2 | x | zy | xcbazy | yzcbax | zyxcba |

Kết quả `zyxcba`: ở `i = 1, j = 0`, ứng viên giữ chiều là `zyx + cba`. Ở `i = 0, j = 0`, ứng viên đảo `zyxabc` đã đạt tiền tố `zyx` nhưng thua `zyxcba` ở ký tự thứ 4 (`a < c`).

In [16]:
def trace_greedy(strs):
    norm = [max(s, s[::-1]) for s in strs]
    print("strs =", strs)
    print("norm =", norm)
    best = "".join(norm)
    print("ứng viên nền best =", repr(best))
    for i, s in enumerate(norm):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        print(f"-- i={i}: s={s!r}, t={t!r}")
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            keep = a + t + b
            flip = b[::-1] + t + a[::-1]
            new_best = max(best, keep, flip)
            mark = "  <-- best mới" if new_best != best else ""
            print(f"   j={j}: a={a!r}, b={b!r} | giữ={keep!r} | đảo={flip!r} | best={new_best!r}{mark}")
            best = new_best
    print("đáp án =", repr(best))
    return best

assert trace_greedy(["abc", "xyz"]) == "zyxcba"

strs = ['abc', 'xyz']
norm = ['cba', 'zyx']
ứng viên nền best = 'cbazyx'
-- i=0: s='cba', t='zyx'
   j=0: a='cba', b='' | giữ='cbazyx' | đảo='zyxabc' | best='zyxabc'  <-- best mới
   j=1: a='ba', b='c' | giữ='bazyxc' | đảo='czyxab' | best='zyxabc'
   j=2: a='a', b='cb' | giữ='azyxcb' | đảo='bczyxa' | best='zyxabc'
-- i=1: s='zyx', t='cba'
   j=0: a='zyx', b='' | giữ='zyxcba' | đảo='cbaxyz' | best='zyxcba'  <-- best mới
   j=1: a='yx', b='z' | giữ='yxcbaz' | đảo='zcbaxy' | best='zyxcba'
   j=2: a='x', b='zy' | giữ='xcbazy' | đảo='yzcbax' | best='zyxcba'
đáp án = 'zyxcba'


Trace thứ hai với `strs = ["a", "bc"]` (kỳ vọng `"cba"`). Điểm mấu chốt: chuỗi `"bc"` phải được chuẩn hóa thành `"cb"`; ứng viên tốt nhất `cba` xuất hiện ngay ở `i = 0` (chuỗi `"a"` chỉ có một ký tự, ứng viên đảo là cắt ngay sau chữ `a`) và cả ở `i = 1`:

| `i` | `s` | `t` | `j` | giữ chiều | đảo chiều | `best` |
|---|---|---|---|---|---|---|
| 0 | a | cb | 0 | acb | cba | cba |
| 1 | cb | a | 0 | cba | abc | cba |
| 1 | cb | a | 1 | bac | cab | cba |

In [17]:
assert trace_greedy(["a", "bc"]) == "cba"
print()
assert trace_greedy(["abc"]) == "cba"

strs = ['a', 'bc']
norm = ['a', 'cb']
ứng viên nền best = 'acb'
-- i=0: s='a', t='cb'
   j=0: a='a', b='' | giữ='acb' | đảo='cba' | best='cba'  <-- best mới
-- i=1: s='cb', t='a'
   j=0: a='cb', b='' | giữ='cba' | đảo='abc' | best='cba'
   j=1: a='b', b='c' | giữ='bac' | đảo='cab' | best='cba'
đáp án = 'cba'

strs = ['abc']
norm = ['cba']
ứng viên nền best = 'cba'
-- i=0: s='cba', t=''
   j=0: a='cba', b='' | giữ='cba' | đảo='abc' | best='cba'
   j=1: a='ba', b='c' | giữ='bac' | đảo='cab' | best='cba'
   j=2: a='a', b='cb' | giữ='acb' | đảo='bca' | best='cba'
đáp án = 'cba'


## 9. Code nộp LeetCode

Lời giải tham lam chuẩn (Cách 2):

In [18]:
from typing import List


class Solution:
    def splitLoopedString(self, strs: List[str]) -> str:
        strs = [max(s, s[::-1]) for s in strs]          # mỗi chuỗi lấy bản lớn hơn
        ans = "".join(strs)
        for i, s in enumerate(strs):
            t = "".join(strs[i + 1:]) + "".join(strs[:i])   # các chuỗi còn lại theo thứ tự vòng
            for j in range(len(s)):
                a, b = s[j:], s[:j]
                ans = max(ans, a + t + b)                  # cắt tại j, giữ chiều
                ans = max(ans, b[::-1] + t + a[::-1])      # cắt tại j, chiều đảo
        return ans


sol = Solution()
for strs, expected in examples:
    got = sol.splitLoopedString(list(strs))
    print(repr(got))
    assert got == expected

'zyxcba'
'cba'


## 10. Vì sao đúng

**Bổ đề 1 — dạng của mọi ứng viên.** Một cách làm hoàn chỉnh gồm: (i) chọn một chiều cho mỗi chuỗi, (ii) chọn điểm cắt. Điểm cắt nằm đúng một vị trí trong vòng, tức nằm **ở đầu** hoặc **bên trong** một chuỗi nào đó `strs[i]` (xét ở chiều đã chọn `o`; chặn "ranh giới" giữa hai chuỗi tính vào đầu chuỗi sau). Nếu cắt tại vị trí `j` của `o`, chuỗi kết quả là `o[j:] + (các chuỗi i+1, ..., n-1, 0, ..., i-1 theo chiều đã chọn) + o[:j]`.

**Bổ đề 2 — các chuỗi còn lại chọn chiều độc lập.** Cố định `(i, o, j)`. Mọi kết quả khả dĩ có cùng độ dài `L`; trong mỗi kết quả, chuỗi `strs[k]` (`k ≠ i`) chiếm **đúng một đoạn vị trí cố định** (độ dài `len(strs[k])`), và các đoạn không giao nhau. So sánh hai kết quả theo từ điển là so từng ký tự từ trái sang phải, tương đương so tuần tự các đoạn từ trái sang phải. Vì đoạn của mỗi chuỗi chỉ phụ thuộc vào chiều của **chính chuỗi đó**, việc chọn chiều của chuỗi này **không ảnh hưởng** các đoạn khác. Do đó lớn nhất toàn cục đạt được khi **từng đoạn lớn nhất**, tức mỗi `strs[k]` chọn `max(strs[k], strs[k][::-1])` (hai bản cùng độ dài nên so từ điển đúng nghĩa).

**Bổ đề 3 — chuỗi chứa điểm cắt phải thử cả hai chiều.** Chuỗi `strs[i]` xuất hiện **tách làm hai đoạn** (đoạn 1 là `o[j:]` đứng đầu và đoạn 3 là `o[:j]` đứng cuối), và cả hai đoạn phụ thuộc vào cùng `o` và `j`. Không có "chiều lớn hơn" tuyệt đối: ví dụ `["acb"]`, chuẩn hóa cho `"bca"`, nhưng đáp án đúng `"cba"` đến từ chiều gốc `"acb"` cắt trước `c`. Vì vậy ta liệt kê cả hai chiều và mọi `j`.

**Kết luận.** Tập ứng viên `{ o[j:] + t + o[:j] }` (với `i`, hai chiều `o`, mọi `j`, `t` là các chuỗi còn lại đã chuẩn hóa) chứa mọi chuỗi tối ưu theo Bổ đề 2 và 3, và mọi phần tử của nó đều là chuỗi hợp lệ theo đề. Vậy lớn nhất trong tập này là đáp án. Cách 3 (cắt tỉa) loại các ứng viên bắt đầu bằng ký tự khác ký tự lớn nhất toàn cục, vốn không thể tối ưu vì luôn có ứng viên bắt đầu bằng ký tự lớn nhất đó. Cách 4 dựng cùng một tập ứng viên bằng cửa sổ trên chuỗi nhân đôi.

In [19]:
# Kiểm chứng Bổ đề 2 bằng thực nghiệm: cố định (i, chiều, j), thử 2^(n-1) kiểu chiều cho các chuỗi còn lại
# và so với việc "chuẩn hóa từng chuỗi". Kết quả lớn nhất phải trùng.
rng = random.Random(10)
for _ in range(300):
    n = rng.randint(2, 5)
    strs = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4))) for _ in range(n)]
    i = rng.randrange(n)
    o = rng.choice([strs[i], strs[i][::-1]])
    j = rng.randrange(len(o))
    others = [strs[k] for k in list(range(i + 1, n)) + list(range(i))]
    best_all = max(o[j:] + "".join(s[::-1] if f else s for s, f in zip(others, flips)) + o[:j]
                   for flips in product((False, True), repeat=len(others)))
    normalized = o[j:] + "".join(max(s, s[::-1]) for s in others) + o[:j]
    assert best_all == normalized, (strs, i, o, j)
print("Bổ đề 2 đúng trên 300 trường hợp ngẫu nhiên")

Bổ đề 2 đúng trên 300 trường hợp ngẫu nhiên


## 11. Time / Space Complexity

Đặt `n` là số chuỗi, `L` là tổng độ dài (`L ≤ 1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn `2^n` kiểu đảo × `L` điểm cắt | `O(2^n · L^2)` | `O(L)` |
| 2. Tham lam chuẩn (nộp) | `O(L^2)` | `O(L)` |
| 3. Cắt tỉa theo ký tự lớn nhất | `O(L^2)` xấu nhất, thường ít hơn | `O(L)` |
| 4. Nhân đôi vòng + cửa sổ | `O(L^2)` | `O(L)` |

Với cách 2: có `2L` ứng viên (mỗi `j` hai chiều), mỗi ứng viên là nối chuỗi tốn `O(L)` và so sánh với `ans` cũng `O(L)`, tổng `O(L^2)`. Dựng `t` cho mỗi `i` tốn `O(L)`, tổng `O(n · L) ⊆ O(L^2)`. Bộ nhớ chỉ giữ vài chuỗi dài `L`, nên `O(L)`. (Đề gốc ký hiệu `n` là độ dài tổng; trùng với `L` ở đây.)

In [20]:
# Đo thực nghiệm ở các hình dạng xấu nhất của L = 1000
shapes = {
    "1 chuỗi 1000 chữ z": ["z" * 1000],
    "1000 chuỗi 1 chữ z": ["z"] * 1000,
    "10 chuỗi x 100 chữ 'z'": ["z" * 100] * 10,
    "ngẫu nhiên 1000 ký tự / 50 chuỗi": None,
}
rng = random.Random(555)
cuts = sorted(rng.sample(range(1, 1000), 49))
pts = [0] + cuts + [1000]
shapes["ngẫu nhiên 1000 ký tự / 50 chuỗi"] = ["".join(rng.choice("abcz") for _ in range(b - a)) for a, b in zip(pts, pts[1:])]

for name, strs in shapes.items():
    assert sum(map(len, strs)) == 1000
    res = []
    for f in (split_greedy, split_pruned, split_doubled):
        t0 = time.perf_counter()
        res.append(f(strs))
        dt = time.perf_counter() - t0
        assert dt < 5
    assert res[0] == res[1] == res[2]
    print(f"{name:36s} -> độ dài kết quả {len(res[0])}, ba cách khớp nhau")

1 chuỗi 1000 chữ z                   -> độ dài kết quả 1000, ba cách khớp nhau
1000 chuỗi 1 chữ z                   -> độ dài kết quả 1000, ba cách khớp nhau
10 chuỗi x 100 chữ 'z'               -> độ dài kết quả 1000, ba cách khớp nhau
ngẫu nhiên 1000 ký tự / 50 chuỗi     -> độ dài kết quả 1000, ba cách khớp nhau


## 12. Bộ test

Chia làm 4 nhóm: (a) ví dụ đề và edge case có đáp án tính tay; (b) phản ví dụ cho các heuristic sai; (c) đối chiếu các cách với **oracle vét cạn** `2^n` trên input nhỏ; (d) test ngẫu nhiên có seed, cả cỡ nhỏ (có oracle) và cỡ lớn đúng ràng buộc (so các cách với nhau).

### 12.1. Test cố định

Mọi đáp án đều tính tay được (xem chú thích), sau đó đối chiếu cả bốn cách và code nộp.

In [21]:
solvers = {
    "brute": split_brute,
    "greedy": split_greedy,
    "pruned": split_pruned,
    "doubled": split_doubled,
    "submit": lambda strs: Solution().splitLoopedString(list(strs)),
}

fixed_cases = [
    (["abc", "xyz"], "zyxcba"),      # ví dụ 1
    (["abc"], "cba"),                # ví dụ 2
    (["a"], "a"),                    # một ký tự
    (["aaa", "aa"], "aaaaa"),        # toàn ký tự giống nhau
    (["z", "a"], "za"),              # chuỗi 1 ký tự: không đảo được gì, cắt trước z
    (["b", "a", "c"], "cba"),        # cắt trước c, vòng lại b a
    (["zz", "a"], "zza"),
    (["ab", "ba"], "bbaa"),          # vòng "abba" cắt trước b thứ nhất: "bba" + "a"
    (["bab"], "bba"),                # cắt tại b cuối: "b" + "ba"
    (["aba"], "baa"),                # cắt tại b, đảo không đổi (đối xứng) -> "baa"
    (["a", "bc"], "cba"),
    (["acb"], "cba"),
    (["b", "ac"], "cba"),
]
for strs, expected in fixed_cases:
    for name, f in solvers.items():
        got = f(list(strs))
        assert got == expected, (name, strs, got, expected)
print("Các test cố định: OK (", len(fixed_cases), "bộ x", len(solvers), "lời giải )")

Các test cố định: OK ( 13 bộ x 5 lời giải )


Giải thích vài đáp án tính tay: với `["ab","ba"]`, giữ nguyên cả hai chuỗi ta được vòng `abba`; cắt trước chữ `b` đầu tiên cho `bba` + `a` = `bbaa`. Đây là ví dụ cho thấy ta **không** nên đưa mọi chuỗi về bản lớn hơn rồi chỉ xoay vòng (vòng chuẩn hóa `baba` chỉ cho tối đa `baba`, nhỏ hơn `bbaa`): chuỗi chứa điểm cắt (`"ab"`) phải được thử cả chiều gốc.

In [22]:
# Kiểm chứng ý vừa nói: vòng chuẩn hóa "baba" chỉ cho tối đa "baba", nhưng đáp án thật là "bbaa"
norm_loop = "".join(max(s, s[::-1]) for s in ["ab", "ba"])
print(norm_loop, "-> xoay lớn nhất:", max(norm_loop[c:] + norm_loop[:c] for c in range(len(norm_loop))))
print("đáp án thật:", Solution().splitLoopedString(["ab", "ba"]))

baba -> xoay lớn nhất: baba
đáp án thật: bbaa


### 12.2. Phản ví dụ cho các heuristic sai

Hai heuristic nghe hợp lý nhưng sai:

- *"Chỉ cần so ký tự đầu và ký tự cuối để quyết định đảo hay không"* (đảo nếu `s[-1] > s[0]`). Sai khi hai đầu bằng nhau nhưng phần giữa khác: `"cabc"` có `c` ở cả hai đầu mà `"cbac" > "cabc"`.
- *"Chỉ thử điểm cắt tại lần xuất hiện đầu tiên của ký tự lớn nhất trong mỗi chuỗi"*. Sai khi các lần xuất hiện sau cho kết quả lớn hơn: `"bab"` chỉ thử vị trí `0` thì được `bab`, trong khi cắt ở `b` cuối cho `bba`.

In [23]:
def wrong_first_last(strs):
    norm = [s[::-1] if s[-1] > s[0] else s for s in strs]       # SAI: so mỗi hai đầu
    ans = "".join(norm)
    for i, s in enumerate(norm):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            ans = max(ans, a + t + b, b[::-1] + t + a[::-1])
    return ans


def wrong_first_top(strs):
    norm = [max(s, s[::-1]) for s in strs]
    top = max(max(s) for s in strs)
    best = ""
    for i, s in enumerate(strs):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        for o in (s, s[::-1]):
            if top in o:
                j = o.index(top)                                   # SAI: chỉ lần xuất hiện đầu tiên
                best = max(best, o[j:] + t + o[:j])
    return best


c1 = ["a", "cabc"]
print("first_last:", wrong_first_last(c1), "| đúng:", split_brute(c1))
assert wrong_first_last(c1) != split_brute(c1)

c2 = ["bab"]
print("first_top :", wrong_first_top(c2), "| đúng:", split_brute(c2))
assert wrong_first_top(c2) != split_brute(c2)
assert Solution().splitLoopedString(c1) == split_brute(c1)
assert Solution().splitLoopedString(c2) == split_brute(c2)

first_last: cacba | đúng: cbaca
first_top : bab | đúng: bba


### 12.3. Đối chiếu với oracle vét cạn (vét hết chuỗi nhỏ)

Liệt kê **mọi** mảng gồm tối đa 3 chuỗi, mỗi chuỗi dài tối đa 3 trên bảng chữ cái `{a, b, c}` (nhiều nghìn trường hợp) và so tất cả các cách với `split_brute`.

In [24]:
def all_strings(max_len, alphabet="abc"):
    out = []
    for l in range(1, max_len + 1):
        out.extend("".join(p) for p in product(alphabet, repeat=l))
    return out

pool = all_strings(3)       # 3 + 9 + 27 = 39 chuỗi
checked = 0
for n in (1, 2, 3):
    arrays = product(pool, repeat=n) if n < 3 else ((a, b, c) for a in pool[:13] for b in pool[:13] for c in pool[:13])
    for strs in arrays:
        strs = list(strs)
        expected = split_brute(strs)
        for name, f in solvers.items():
            assert f(list(strs)) == expected, (name, strs, f(list(strs)), expected)
        checked += 1
print("Vét hết các mảng nhỏ:", checked, "mảng, mọi cách đều khớp oracle")

Vét hết các mảng nhỏ: 3757 mảng, mọi cách đều khớp oracle


### 12.4. Test ngẫu nhiên có seed

Nhóm 1: cỡ nhỏ (`n ≤ 6`, độ dài ≤ 5, bảng chữ `abc` để có nhiều trùng lặp) so với oracle. Nhóm 2: cỡ lớn đúng ràng buộc (tổng độ dài tới `1000`) so các cách 2, 3, 4 với nhau và với bản "nộp".

In [25]:
rng = random.Random(555)
for _ in range(2000):
    n = rng.randint(1, 6)
    alphabet = rng.choice(["ab", "abc", "abcz"])
    strs = ["".join(rng.choice(alphabet) for _ in range(rng.randint(1, 5))) for _ in range(n)]
    expected = split_brute(strs)
    for name, f in solvers.items():
        assert f(list(strs)) == expected, (name, strs)
print("Ngẫu nhiên cỡ nhỏ (seed 555): 2000 mảng, khớp oracle")

Ngẫu nhiên cỡ nhỏ (seed 555): 2000 mảng, khớp oracle


In [26]:
rng = random.Random(2026)
for case in range(150):
    L = rng.randint(1, 1000)
    n = rng.randint(1, min(L, 40))
    cuts = sorted(rng.sample(range(1, L), n - 1)) if n > 1 else []
    pts = [0] + cuts + [L]
    alphabet = rng.choice(["a", "ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    strs = ["".join(rng.choice(alphabet) for _ in range(b - a)) for a, b in zip(pts, pts[1:])]
    assert 1 <= len(strs) <= 1000 and sum(map(len, strs)) == L and all(1 <= len(s) <= 1000 for s in strs)
    r2 = split_greedy(strs)
    assert r2 == split_pruned(strs) == split_doubled(strs) == Solution().splitLoopedString(list(strs))
    assert len(r2) == L
print("Ngẫu nhiên cỡ lớn (seed 2026): 150 mảng, các cách khớp nhau")

Ngẫu nhiên cỡ lớn (seed 2026): 150 mảng, các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên chuẩn hóa các chuỗi còn lại

Không đảo các chuỗi khác về bản lớn hơn nghĩa là bỏ lỡ cơ hội làm kết quả lớn hơn. Ví dụ `["a", "bc"]`: ta cần đảo `"bc"` thành `"cb"` để ra `"cba"`.

In [27]:
def wrong_no_normalize(strs):
    ans = "".join(strs)
    for i, s in enumerate(strs):
        t = "".join(strs[i + 1:]) + "".join(strs[:i])      # SAI: t không được chuẩn hóa
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            ans = max(ans, a + t + b, b[::-1] + t + a[::-1])
    return ans

print("sai:", wrong_no_normalize(["a", "bc"]), "| đúng:", Solution().splitLoopedString(["a", "bc"]))
assert wrong_no_normalize(["a", "bc"]) == "cab" and Solution().splitLoopedString(["a", "bc"]) == "cba"

sai: cab | đúng: cba


### Lỗi 2 — Chỉ thử một chiều của chuỗi chứa điểm cắt

Nếu chỉ thử `a + t + b` (chuỗi đã chuẩn hóa, giữ chiều) mà bỏ ứng viên chiều đảo `b[::-1] + t + a[::-1]` thì sai: `["acb"]` đáp án đúng `"cba"` nhưng chỉ xoay vòng của `"bca"` thì tối đa `"cab"`.

In [28]:
def wrong_one_orientation(strs):
    strs = [max(s, s[::-1]) for s in strs]
    ans = "".join(strs)
    for i, s in enumerate(strs):
        t = "".join(strs[i + 1:]) + "".join(strs[:i])
        for j in range(len(s)):
            ans = max(ans, s[j:] + t + s[:j])              # SAI: thiếu chiều đảo
    return ans

print("sai:", wrong_one_orientation(["acb"]), "| đúng:", Solution().splitLoopedString(["acb"]))
assert wrong_one_orientation(["acb"]) == "cab" and Solution().splitLoopedString(["acb"]) == "cba"

sai: cab | đúng: cba


### Lỗi 3 — Quên đoạn "vòng lại" `strs[:i]`

Chuỗi kết quả phải đi hết **một vòng**, nên sau `strs[i+1:]` còn phải nối tiếp `strs[:i]`. Nếu bỏ `strs[:i]`, độ dài kết quả sai (thiếu ký tự) và có thể ra đáp án nhỏ hơn hoặc ngắn hơn. Ví dụ `["b", "ac"]` kỳ vọng `"cba"`.

In [29]:
def wrong_no_wrap(strs):
    strs = [max(s, s[::-1]) for s in strs]
    ans = "".join(strs)
    for i, s in enumerate(strs):
        t = "".join(strs[i + 1:])                          # SAI: thiếu "".join(strs[:i])
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            ans = max(ans, a + t + b, b[::-1] + t + a[::-1])
    return ans

got = wrong_no_wrap(["b", "ac"])
print("sai:", got, "| đúng:", Solution().splitLoopedString(["b", "ac"]))
assert got != "cba" and Solution().splitLoopedString(["b", "ac"]) == "cba"

sai: cab | đúng: cba


### Lỗi 4 — Chỉ cắt ở ranh giới giữa các chuỗi

Cắt vòng ở bất kỳ điểm nào, **kể cả giữa một chuỗi**. Nếu chỉ thử bắt đầu tại đầu mỗi chuỗi (`strs[i:] + strs[:i]`), sẽ bỏ lỡ nhiều đáp án: `["aba"]` đáp án `"baa"` cắt giữa chuỗi, còn bắt đầu từ ranh giới chỉ cho `"aba"`.

In [30]:
def wrong_boundary_only(strs):
    strs = [max(s, s[::-1]) for s in strs]
    return max("".join(strs[i:] + strs[:i]) for i in range(len(strs)))      # SAI: chỉ cắt ở ranh giới

print("sai:", wrong_boundary_only(["aba"]), "| đúng:", Solution().splitLoopedString(["aba"]))
assert wrong_boundary_only(["aba"]) == "aba" and Solution().splitLoopedString(["aba"]) == "baa"

sai: aba | đúng: baa


### Lỗi 5 — Dùng `reversed(s)` thay cho `s[::-1]`

`reversed(s)` trả về một **iterator** (đối tượng duyệt ngược), không phải chuỗi. So sánh `reversed(s) > s` hoặc nối `+` với chuỗi sẽ báo `TypeError`. Muốn chuỗi đảo hãy dùng `s[::-1]` hoặc `''.join(reversed(s))`.

In [31]:
s = "abc"
print(type(reversed(s)), type(s[::-1]))
try:
    print(max(s, reversed(s)))
except TypeError as e:
    print("TypeError:", e)
print("đúng:", max(s, s[::-1]), "|", "".join(reversed(s)))

<class 'reversed'> <class 'str'>
TypeError: '>' not supported between instances of 'reversed' and 'str'
đúng: cba | cba


## 14. Mẹo & nhận dạng pattern

- **"Chuỗi lớn nhất/nhỏ nhất theo từ điển" + "mỗi phần tử có hai lựa chọn độc lập"** → nghĩ ngay đến **chọn tham lam từng phần**, nếu mỗi phần chiếm vị trí cố định trong kết quả. Hãy kiểm tra "độ dài kết quả cố định" trước.
- **Vòng tròn / xoay vòng** → nhân đôi chuỗi (`s + s`) rồi lấy cửa sổ độ dài `L`; hoặc dùng `s[c:] + s[:c]`.
- **Điểm đặc biệt (điểm cắt) phá vỡ tính độc lập**: chuỗi chứa điểm cắt bị tách thành hai đoạn ở hai đầu kết quả nên phải **liệt kê** chiều và vị trí của nó, trong khi phần còn lại tham lam.
- **Kết quả lớn nhất phải bắt đầu bằng ký tự lớn nhất có thể** → cắt tỉa ứng viên (cách 3). Cùng ý với *Last Substring in Lexicographical Order* (LeetCode 1163).
- Với `so sánh chuỗi cùng độ dài` mới có thể "so từng đoạn riêng"; nếu độ dài kết quả thay đổi (ví dụ nối thứ tự tùy ý như *Largest Number*, LeetCode 179) thì phải dùng bộ so sánh `a + b > b + a`.
- Đừng viết `reversed(s)` khi cần chuỗi; dùng `s[::-1]`.
- Luôn có oracle vét cạn `2^n` cho `n` nhỏ để kiểm chứng lời giải tham lam.

## 15. Biến thể

### 15.1. Không được đảo chuỗi (chỉ nối vòng rồi cắt)

Khi không được đảo, vòng cố định là `"".join(strs)`; lớn nhất trong các phép xoay là cửa sổ lớn nhất trên chuỗi nhân đôi. Đây là bài **"xoay vòng lớn nhất"** đơn giản.

In [32]:
def largest_rotation(strs):
    loop = "".join(strs)
    doubled = loop + loop
    L = len(loop)
    return max(doubled[c:c + L] for c in range(L))

print(largest_rotation(["abc", "xyz"]))     # zabcxy
print(largest_rotation(["ba", "ab"]))       # vòng "baab", xoay lớn nhất là bbaa
assert largest_rotation(["abc", "xyz"]) == "zabcxy"
assert largest_rotation(["ba", "ab"]) == "bbaa"

zabcxy
bbaa


### 15.2. Trả về cả "cách cắt" tối ưu

Nếu đề yêu cầu cho biết **chuỗi nào chứa điểm cắt, vị trí cắt và chiều đảo**, ta chỉ cần lưu lại `(i, j, chiều)` mỗi khi cập nhật `best`.

In [33]:
def split_with_info(strs):
    norm = [max(s, s[::-1]) for s in strs]
    best, info = "".join(norm), (0, 0, "giữ")
    for i, s in enumerate(norm):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            if a + t + b > best:
                best, info = a + t + b, (i, j, "giữ")
            if b[::-1] + t + a[::-1] > best:
                best, info = b[::-1] + t + a[::-1], (i, j, "đảo")
    return best, info

print(split_with_info(["abc", "xyz"]))     # ('zyxcba', (1, 0, 'giữ'))
assert split_with_info(["abc", "xyz"]) == ("zyxcba", (1, 0, "giữ"))

('zyxcba', (1, 0, 'giữ'))


### 15.3. Bản "nhỏ nhất theo từ điển"

Đổi `max` thành `min` ở **mọi** chỗ (chuẩn hóa mỗi chuỗi lấy bản nhỏ hơn, cắt tìm chuỗi nhỏ nhất). Lập luận đối xứng. Đây là bài tập 5 ở mục 17.

### 15.4. Các biến thể khác (chỉ mô tả)

- **Cho phép hoán vị thứ tự các chuỗi**: khó hơn nhiều; mỗi chuỗi nên lấy bản lớn hơn rồi sắp xếp bằng bộ so sánh `a + b > b + a` (như *Largest Number*, 179), sau đó vẫn phải thử điểm cắt.
- **Chỉ đảo tối đa `k` chuỗi**: không còn độc lập hoàn toàn; cần quy hoạch động hoặc chọn `k` chuỗi mang lại lợi ích lớn nhất.
- **Bài liên quan**: Last Substring in Lexicographical Order (1163), Largest Number (179), Lexicographically Smallest String After Operations (có thể có biến thể xoay/đảo), Rotate String (796).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử `2^n` kiểu đảo × `L` điểm cắt | `O(2^n · L^2)` | `O(L)` | chỉ làm oracle kiểm chứng với `n` nhỏ |
| 2. Tham lam chuẩn | chuẩn hóa các chuỗi còn lại, thử mọi điểm cắt ở hai chiều | `O(L^2)` | `O(L)` | **cách nên nộp**: ngắn, rõ, đủ nhanh |
| 3. Cắt tỉa | chỉ dựng ứng viên bắt đầu bằng ký tự lớn nhất toàn cục | `O(L^2)` xấu nhất | `O(L)` | khi cần nhanh hơn trên dữ liệu ngẫu nhiên; ý tưởng nên nhớ |
| 4. Nhân đôi vòng | dựng vòng một lần cho mỗi (chuỗi, chiều) rồi cắt cửa sổ | `O(L^2)` | `O(L)` | khi muốn diễn đạt đúng "vòng tròn", dễ mở rộng sang biến thể xoay |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print("abc"[::-1])
print(max("abd", "abcz"))
print("xyz"[1:] + "xyz"[:1])
```

In [34]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `strs = ["ab", "c"]`: (a) chuẩn hóa từng chuỗi; (b) liệt kê **mọi** ứng viên của thuật toán tham lam (`i`, `j`, giữ chiều / đảo chiều); (c) cho biết đáp án.

In [35]:
# Bài 2: điền kết quả tính tay
hand_norm = None        # danh sách sau chuẩn hóa, ví dụ ["..", ".."]
hand_answer = None      # đáp án cuối cùng (chuỗi)

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo khuôn mẫu tham lam.

In [36]:
def exercise_3_split(strs: List[str]) -> str:
    # TODO 1: chuẩn hóa: mỗi chuỗi lấy max(s, s[::-1])
    norm = None
    ans = ""
    # TODO 2: duyệt i, s trong norm; t = các chuỗi còn lại theo thứ tự vòng (norm[i+1:] rồi norm[:i])
    #     TODO 3: duyệt j trong range(len(s)); a = s[j:], b = s[:j]
    #         TODO 4: cập nhật ans với a + t + b và b[::-1] + t + a[::-1]
    return ans


# assert exercise_3_split(["abc", "xyz"]) == "zyxcba"
# assert exercise_3_split(["abc"]) == "cba"
# assert exercise_3_split(["a", "bc"]) == "cba"

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại các cách trên, hãy tự viết `exercise_4_split_doubled(strs)` theo cách **nhân đôi vòng + cửa sổ** (cách 4): với mỗi chuỗi và mỗi chiều dựng vòng một lần.

In [37]:
def exercise_4_split_doubled(strs: List[str]) -> str:
    # TODO
    pass


# assert exercise_4_split_doubled(["abc", "xyz"]) == "zyxcba"
# assert exercise_4_split_doubled(["aba"]) == "baa"
# assert exercise_4_split_doubled(["ab", "ba"]) == "bbaa"

### Bài 5 — Mini challenge: bản "nhỏ nhất theo từ điển" (khó)

Cùng luật chơi (đảo hoặc không đảo từng chuỗi, nối thành vòng theo thứ tự, cắt tại một điểm), nhưng yêu cầu **chuỗi nhỏ nhất** theo từ điển. Viết `exercise_5_smallest(strs)` trả về chuỗi đó.

```python
exercise_5_smallest(["abc", "xyz"]) -> "abcxyz"
exercise_5_smallest(["b"]) -> "b"
exercise_5_smallest(["ba", "z"]) -> "abz"
```

Gợi ý: nghĩ xem đối xứng với bản "lớn nhất" thế nào; chú ý ứng viên nền ban đầu.

In [38]:
def exercise_5_smallest(strs: List[str]) -> str:
    # TODO
    pass


# assert exercise_5_smallest(["abc", "xyz"]) == "abcxyz"
# assert exercise_5_smallest(["b"]) == "b"
# assert exercise_5_smallest(["ba", "z"]) == "abz"

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `[::-1]` đảo chuỗi; `max` trên chuỗi so theo từ điển; `s[1:] + s[:1]` là xoay trái 1 vị trí.
- Mức 2: so `"abd"` và `"abcz"`: hai ký tự đầu `ab` giống nhau, ký tự thứ ba `d` và `c` khác.
- Mức 3: dòng 1 `cba`; dòng 2 `abd`; dòng 3 `yzx`.

**Bài 2**
- Mức 1: chuẩn hóa `"ab"` → so `"ab"` với `"ba"` → lấy `"ba"`; `"c"` giữ nguyên.
- Mức 2: `norm = ["ba", "c"]`. Với `i = 0`: `t = "c"`, `j = 0, 1`; với `i = 1`: `t = "ba"`, `j = 0`.
- Mức 3: các ứng viên gồm `bac`, `cab` (`i=0,j=0`), `acb`, `bca` (`i=0,j=1`), `cba`, `bac` (`i=1,j=0`); lớn nhất `cba`.

**Bài 3**
- Mức 1: `norm = [max(s, s[::-1]) for s in strs]`.
- Mức 2: `t = "".join(norm[i + 1:]) + "".join(norm[:i])`.
- Mức 3: với mỗi `j`, `ans = max(ans, a + t + b, b[::-1] + t + a[::-1])`.

**Bài 4**
- Mức 1: với chuỗi `i`, `before = join(norm[:i])`, `after = join(norm[i+1:])`; với `o` trong `(s, s[::-1])`: `loop = before + o + after`.
- Mức 2: `doubled = loop + loop`; điểm cắt trong chuỗi `i` bắt đầu tại `len(before) + j`.
- Mức 3: `best = max(best, doubled[start : start + L])` với `L = len(loop)`; `len(before)` bằng tổng độ dài của `strs[:i]`.

**Bài 5**
- Mức 1: thay `max` bằng `min` ở mọi chỗ.
- Mức 2: chuẩn hóa `norm = [min(s, s[::-1]) ...]`; ứng viên nền `"".join(norm)`; so sánh dùng `min`.
- Mức 3: vòng lặp giống hệt cách 2: `ans = min(ans, a + t + b, b[::-1] + t + a[::-1])`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [39]:
print("abc"[::-1])
print(max("abd", "abcz"))
print("xyz"[1:] + "xyz"[:1])

assert "abc"[::-1] == "cba"
assert max("abd", "abcz") == "abd"
assert "xyz"[1:] + "xyz"[:1] == "yzx"
print("Bài 1 OK")

cba
abd
yzx
Bài 1 OK


### Đáp án Bài 2

`norm = ["ba", "c"]`. Ứng viên: `(i=0, j=0)`: `bac` / `cab`; `(i=0, j=1)`: `acb` / `bca`; `(i=1, j=0)`: `cba` / `bac`. Lớn nhất `cba`.

In [40]:
norm2 = [max(s, s[::-1]) for s in ["ab", "c"]]
cands = []
for i, s in enumerate(norm2):
    t = "".join(norm2[i + 1:]) + "".join(norm2[:i])
    for j in range(len(s)):
        a, b = s[j:], s[:j]
        cands.append((i, j, a + t + b, b[::-1] + t + a[::-1]))
for c in cands:
    print(c)

assert norm2 == ["ba", "c"]
assert max(max(c[2], c[3]) for c in cands) == "cba"
assert Solution().splitLoopedString(["ab", "c"]) == split_brute(["ab", "c"]) == "cba"
print("Bài 2 OK")

(0, 0, 'bac', 'cab')
(0, 1, 'acb', 'bca')
(1, 0, 'cba', 'bac')
Bài 2 OK


### Đáp án Bài 3

In [41]:
def exercise_3_solution(strs: List[str]) -> str:
    norm = [max(s, s[::-1]) for s in strs]
    ans = ""
    for i, s in enumerate(norm):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            ans = max(ans, a + t + b, b[::-1] + t + a[::-1])
    return ans


assert exercise_3_solution(["abc", "xyz"]) == "zyxcba"
assert exercise_3_solution(["abc"]) == "cba"
assert exercise_3_solution(["a", "bc"]) == "cba"
rng = random.Random(3)
for _ in range(500):
    strs = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 5))]
    assert exercise_3_solution(strs) == split_brute(strs)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_solution(strs: List[str]) -> str:
    norm = [max(s, s[::-1]) for s in strs]
    best = ""
    for i, s in enumerate(strs):
        before = "".join(norm[:i])
        after = "".join(norm[i + 1:])
        for o in (s, s[::-1]):
            loop = before + o + after
            doubled = loop + loop
            L = len(loop)
            for j in range(len(o)):
                start = len(before) + j
                best = max(best, doubled[start:start + L])
    return best


assert exercise_4_solution(["abc", "xyz"]) == "zyxcba"
assert exercise_4_solution(["aba"]) == "baa"
assert exercise_4_solution(["ab", "ba"]) == "bbaa"
rng = random.Random(4)
for _ in range(500):
    strs = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 5))]
    assert exercise_4_solution(strs) == split_brute(strs)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [43]:
def exercise_5_solution(strs: List[str]) -> str:
    norm = [min(s, s[::-1]) for s in strs]
    ans = "".join(norm)
    for i, s in enumerate(norm):
        t = "".join(norm[i + 1:]) + "".join(norm[:i])
        for j in range(len(s)):
            a, b = s[j:], s[:j]
            ans = min(ans, a + t + b, b[::-1] + t + a[::-1])
    return ans


def brute_smallest(strs):
    best = None
    for flips in product((False, True), repeat=len(strs)):
        loop = "".join(s[::-1] if f else s for s, f in zip(strs, flips))
        for c in range(len(loop)):
            cand = loop[c:] + loop[:c]
            if best is None or cand < best:
                best = cand
    return best


assert exercise_5_solution(["abc", "xyz"]) == "abcxyz"
assert exercise_5_solution(["b"]) == "b"
assert exercise_5_solution(["ba", "z"]) == "abz"
rng = random.Random(5)
for _ in range(1000):
    strs = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 5))]
    assert exercise_5_solution(strs) == brute_smallest(strs), strs
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def splitLoopedString(self, strs: List[str]) -> str:
        strs = [max(s, s[::-1]) for s in strs]               # mỗi chuỗi lấy bản lớn hơn
        ans = "".join(strs)
        for i, s in enumerate(strs):
            t = "".join(strs[i + 1:]) + "".join(strs[:i])    # các chuỗi còn lại theo thứ tự vòng
            for j in range(len(s)):
                a, b = s[j:], s[:j]                          # điểm cắt tại j trong strs[i]
                ans = max(ans, a + t + b)                    # chiều giữ nguyên
                ans = max(ans, b[::-1] + t + a[::-1])        # chiều đảo
        return ans
```

Câu thần chú: **"chuỗi nào không chứa điểm cắt thì lấy bản lớn hơn; chuỗi chứa điểm cắt thì thử mọi vị trí cắt ở cả hai chiều."**

### Checklist trước khi submit

- [ ] Chuẩn hóa **trước**: `max(s, s[::-1])` cho mọi chuỗi (dùng `s[::-1]`, không dùng `reversed`).
- [ ] `t` gồm cả `strs[i+1:]` **và** `strs[:i]` (phần vòng lại), đúng thứ tự.
- [ ] Thử **cả hai chiều** của chuỗi chứa điểm cắt: `a + t + b` và `b[::-1] + t + a[::-1]`.
- [ ] Điểm cắt đi qua **mọi** `j` trong chuỗi, không chỉ ranh giới giữa các chuỗi.
- [ ] Khởi tạo `ans = "".join(strs)` (hoặc chuỗi rỗng; các ứng viên cùng độ dài nên so sánh được).
- [ ] Test: `["abc","xyz"]`, `["abc"]`, một ký tự, chuỗi đối xứng (`"aba"`), `["ab","ba"]`, toàn ký tự giống nhau.
- [ ] Đúng skeleton: `class Solution`, method `splitLoopedString(self, strs: List[str]) -> str`.
- [ ] Bài Premium: nếu có tài khoản, đối chiếu lại đề gốc và ví dụ.